### 보드라이프 게임 데이터 전처리

**대상 파일**: `boardlife_games_all.csv` (3322행, 17컬럼)  
**출력 파일**: `boardlife_games_final.csv`

**전처리 항목**
1. `detail_url` 컬럼 삭제 → 원본에 존재하지 않음 (해당 없음)
2. 범위형 문자열 컬럼 분리: `players`, `recommended_players`, `playing_time` → 최소/최대값 분리, 없는 값은 NaN
3. 연령 컬럼 정수형 변환: `'14세 이상'` → `14`
4. 다중 태그 컬럼 파싱: `designer`, `category`, `mechanism` → `|` 구분 문자열(리스트)
5. 쉼표 기준 split 후 trim 처리
6. `category_rank` → 딕셔너리 형태로 정규화 (JSON 문자열로 저장)

In [2]:
import pandas as pd
import re
import json

games = pd.read_csv('data/boardlife_games_all.csv')

print('shape:', games.shape)
print('columns:', games.columns.tolist())

shape: (3322, 17)
columns: ['rank', 'title', 'title_eng', 'players', 'recommended_players', 'playing_time', 'age', 'weight', 'designer', 'artist', 'description', 'type', 'category', 'mechanism', 'image', 'avg_rating', 'category_rank']


---
## 0. title_eng 이상값 처리

영문명이 없는 게임은 `'-'`으로 저장되어 있었다. NaN으로 처리한다.

In [3]:
games['title_eng'] = games['title_eng'].apply(
    lambda x: None if pd.isna(x) or str(x).strip() == '-' else x
)
print('title_eng NaN 개수:', games['title_eng'].isnull().sum())

title_eng NaN 개수: 1


---
#### 1. players 파싱 → min_players, max_players

기존 `players` 컬럼은 `'2-4명'` 형태의 문자열로 저장되어 있었다.  
숫자 비교 및 필터링이 가능하도록 최소/최대 인원 컬럼으로 분리한다.  
없는 값(`'-'`)은 NaN 처리.  
비정상값(max=0, min>100, min>max, 파싱 불가)도 NaN 처리.

| 기존 값 | min_players | max_players | 비고 |
| --- | --- | --- | --- |
| `2-4명` | 2 | 4 | 정상 |
| `4명` | 4 | 4 | 정상 |
| `-` | NaN | NaN | 결측 |
| `2-0명` | NaN | NaN | 비정상 (크롤링 오류) |
| `2025-2명` | NaN | NaN | 비정상 (크롤링 오류) |

In [4]:
def parse_players(val):
    if pd.isna(val) or str(val).strip() == '-':
        return None, None
    val = str(val).replace('명', '').replace(' ', '')
    if '-' in val:
        parts = val.split('-')
        try:
            mn, mx = int(parts[0]), int(parts[1])
        except ValueError:
            return None, None
        # 비정상 범위 NaN 처리
        if mx == 0 or mn > 100 or mn > mx:
            return None, None
        return mn, mx
    else:
        try:
            v = int(val)
        except ValueError:
            return None, None
        return v, v

games[['min_players', 'max_players']] = games['players'].apply(
    lambda x: pd.Series(parse_players(x))
)
games = games.drop(columns=['players'])

print('min_players 샘플:', games['min_players'].dropna().astype(int).unique()[:8])
print('max_players 샘플:', games['max_players'].dropna().astype(int).unique()[:8])
print('NaN 개수:', games['min_players'].isnull().sum())

min_players 샘플: [2 1 3 4 6 5 7 8]
max_players 샘플: [ 4  2  5  7  8  6 21 10]
NaN 개수: 15


---
#### 2. recommended_players 파싱 → best_players, recommend_players

`recommended_players` 컬럼은 `'베스트:4인 , 추천:2인'` 형태로 저장되어 있었다.  
베스트 인원(`best_players`)과 추천 인원(`recommend_players`)을 각각 숫자로 추출한다.  
없는 값(`'투표 정보 없음'`, `'2+인'` 등)은 NaN 처리.

| 기존 값 | best_players | recommend_players |
| --- | --- | --- |
| `베스트:4인 , 추천:2인` | 4 | 2 |
| `베스트:3인` | 3 | NaN |
| `투표 정보 없음` | NaN | NaN |

In [5]:
def parse_recommended(val):
    if pd.isna(val) or '투표' in str(val):
        return None, None
    best, rec = None, None
    m = re.search(r'베스트:(\d+)', str(val))
    if m:
        best = int(m.group(1))
    m = re.search(r'추천:(\d+)', str(val))
    if m:
        rec = int(m.group(1))
    return best, rec

games[['best_players', 'recommend_players']] = games['recommended_players'].apply(
    lambda x: pd.Series(parse_recommended(x))
)
games = games.drop(columns=['recommended_players'])

print('best_players 샘플:', games['best_players'].dropna().astype(int).unique()[:10])
print('recommend_players 샘플:', games['recommend_players'].dropna().astype(int).unique()[:10])
print('best_players NaN 개수:', games['best_players'].isnull().sum())
print('recommend_players NaN 개수:', games['recommend_players'].isnull().sum())

best_players 샘플: [ 4  3  2  1  5  6 11  8  7 12]
recommend_players 샘플: [ 2  3  1  4  5  7  6 11 10  9]
best_players NaN 개수: 306
recommend_players NaN 개수: 391


---
#### 3. playing_time 파싱 → min_time, max_time

기존 `playing_time` 컬럼은 `'60-120분'` 형태의 문자열로 저장되어 있었다.  
최소/최대 플레이 시간으로 분리하고, `'분'` 제거 후 정수형으로 변환한다.  
없는 값(`'-'`)은 NaN 처리.  
비정상값(max=0, min>max, 파싱 불가)도 NaN 처리.

| 기존 값 | min_time | max_time | 비고 |
| --- | --- | --- | --- |
| `30-60분` | 30 | 60 | 정상 |
| `90분` | 90 | 90 | 정상 |
| `-` | NaN | NaN | 결측 |
| `20-0분` | NaN | NaN | 비정상 (크롤링 오류) |

In [6]:
def parse_time(val):
    if pd.isna(val) or str(val).strip() == '-':
        return None, None
    val = str(val).replace('분', '').replace(' ', '')
    if '-' in val:
        parts = val.split('-')
        try:
            mn, mx = int(parts[0]), int(parts[1])
        except ValueError:
            return None, None
        # 비정상 범위 NaN 처리
        if mx == 0 or mn > mx:
            return None, None
        return mn, mx
    else:
        try:
            v = int(val)
        except ValueError:
            return None, None
        return v, v

games[['min_time', 'max_time']] = games['playing_time'].apply(
    lambda x: pd.Series(parse_time(x))
)
games = games.drop(columns=['playing_time'])

print('min_time 샘플:', games['min_time'].dropna().astype(int).unique()[:8])
print('max_time 샘플:', games['max_time'].dropna().astype(int).unique()[:8])
print('NaN 개수:', games['min_time'].isnull().sum())

min_time 샘플: [ 60  40 120  30  90 180 150  20]
max_time 샘플: [120 160  60 150 180  90  30 240]
NaN 개수: 28


---
#### 4. age 정수형 변환

`'14세 이상'` → `14`. 없는 값(`'-'`)은 NaN 처리.

| 기존 값 | 변환 후 |
| --- | --- |
| `8세 이상` | 8 |
| `14세 이상` | 14 |
| `-` | NaN |

In [7]:
def parse_age(val):
    if pd.isna(val) or str(val).strip() == '-':
        return None
    m = re.search(r'\d+', str(val))
    return int(m.group()) if m else None

games['age'] = games['age'].apply(parse_age)

print('age 샘플:', games['age'].dropna().astype(int).unique())
print('NaN 개수:', games['age'].isnull().sum())

age 샘플: [14 13 12 10 15  8 18  7  9 16 17  6 11  5  4 19 20  3]
NaN 개수: 21


---
#### 5. weight float 변환

문자열 `'3.87'` → float. 없는 값(`'-'`)은 NaN 처리.

In [8]:
games['weight'] = games['weight'].apply(
    lambda x: None if pd.isna(x) or str(x).strip() == '-' else float(x)
)

print('weight 샘플:', games['weight'].dropna().unique()[:8])
print('NaN 개수:', games['weight'].isnull().sum())

weight 샘플: [3.87 3.79 3.91 2.83 4.44 4.4  4.3  2.97]
NaN 개수: 211


---
#### 5-1. artist, type 이상값 처리

`artist`, `type` 컬럼에 `'정보없음'` 값이 존재한다. NaN으로 처리한다.

| 기존 값 | 변환 후 |
| --- | --- |
| `정보없음` | NaN |
| `-` | NaN |
| `없음` | NaN |

In [15]:
games['artist'] = games['artist'].apply(
    lambda x: None if pd.isna(x) or str(x).strip() in ('정보없음', '-', '없음', '') else x
)
games['type'] = games['type'].apply(
    lambda x: None if pd.isna(x) or str(x).strip() in ('정보없음', '-', '없음', '') else x
)

print('artist NaN 개수:', games['artist'].isnull().sum())
print('type NaN 개수:', games['type'].isnull().sum())

artist NaN 개수: 402
type NaN 개수: 616


---
#### 6. 다중 태그 컬럼 파싱 → 리스트 정규화

`designer`, `category`, `mechanism` 컬럼은 여러 값이 쉼표(`,`)로 구분된 문자열로 저장되어 있었다.  
쉼표 기준으로 split 후 각 항목 앞뒤 공백(trim) 처리한다.  
CSV 저장을 위해 `|` 구분자로 재결합한다.

```
'산업 / 제조, 경제, 운송'       → '산업 / 제조|경제|운송'
'Martin Wallace , Matt Tolman' → 'Martin Wallace|Matt Tolman'
```

In [10]:
def parse_list_col(val):
    if pd.isna(val) or str(val).strip() in ('정보없음', '-', '없음', ''):
        return None
    items = [item.strip() for item in str(val).split(',') if item.strip()]
    return '|'.join(items) if items else None

for col in ['designer', 'category', 'mechanism']:
    games[col] = games[col].apply(parse_list_col)

print('designer 샘플:')
print(games['designer'].head(3).to_string())
print()
print('category 샘플:')
print(games['category'].head(3).to_string())
print()
print('mechanism 샘플:')
print(games['mechanism'].head(2).to_string())

designer 샘플:
0    Martin Wallace|Matt Tolman|Gavan Brown
1                               Tomáš Holek
2                            Isaac Childres

category 샘플:
0    산업 / 제조|나폴레옹 시대 이후|운송|이성 시대|경제|기차
1                       SF 공상 과학|우주 탐험
2                    모험|탐험|판타지|전투|미니어처

mechanism 샘플:
0    핸드 관리|네트워크 및 경로 구축|수입|변화하는 게임 세팅|테크 트리 / 테크 트랙...
1    게임 종료 보너스|수입|솔로/솔로테어 게임|변화하는 게임 세팅|차례 순서: 정방향|...


---
## 7. category_rank 딕셔너리 정규화

`category_rank` 컬럼은 `'전략 순위 3위 | 테마 순위 1위'` 형태로 저장되어 있었다.  
`|` 기준으로 분리 후 `'{카테고리} 순위 {숫자}위'` 패턴으로 파싱하여 딕셔너리로 변환한다.  
CSV 저장을 위해 JSON 문자열로 직렬화한다.

```
'전략 순위 3위 | 테마 순위 1위' → '{"전략": 3, "테마": 1}'
'전략 순위 1위'                 → '{"전략": 1}'
```

카테고리 종류: `전략`, `테마`, `워게임`, `가족`, `어린이`, `추상`, `컬렉터블`, `파티`

In [11]:
def parse_category_rank(val):
    if pd.isna(val):
        return None
    result = {}
    for part in str(val).split('|'):
        m = re.match(r'(.+?)\s*순위\s*(\d+)위', part.strip())
        if m:
            result[m.group(1).strip()] = int(m.group(2))
    return json.dumps(result, ensure_ascii=False) if result else None

games['category_rank'] = games['category_rank'].apply(parse_category_rank)

print('샘플:')
print(games['category_rank'].dropna().head(5).to_string())
print()
print('NaN 개수:', games['category_rank'].isnull().sum())

샘플:
0             {"전략": 1}
1             {"전략": 2}
2    {"전략": 3, "테마": 1}
3    {"전략": 4, "테마": 2}
4             {"전략": 5}

NaN 개수: 619


---
#### 8. 최종 확인

In [12]:
print('shape:', games.shape)
print('columns:', games.columns.tolist())
print()
print('dtypes:')
print(games.dtypes)
print()
print('NaN 현황:')
print(games.isnull().sum())
print()
# 이상값 잔존 확인
print('min_players 최대값:', games['min_players'].max())
print('max_players 최솟값 (0 제외):', games.loc[games['max_players'] > 0, 'max_players'].min())
print('max_time 최솟값 (0 제외):', games.loc[games['max_time'] > 0, 'max_time'].min())

shape: (3322, 20)
columns: ['rank', 'title', 'title_eng', 'age', 'weight', 'designer', 'artist', 'description', 'type', 'category', 'mechanism', 'image', 'avg_rating', 'category_rank', 'min_players', 'max_players', 'best_players', 'recommend_players', 'min_time', 'max_time']

dtypes:
rank                   int64
title                    str
title_eng                str
age                  float64
weight               float64
designer                 str
artist                   str
description              str
type                     str
category                 str
mechanism                str
image                    str
avg_rating           float64
category_rank            str
min_players          float64
max_players          float64
best_players         float64
recommend_players    float64
min_time             float64
max_time             float64
dtype: object

NaN 현황:
rank                   0
title                  0
title_eng              1
age                   21
weight      

In [13]:
games.head(3)

,rank,title,title_eng,age,weight,designer,artist,description,type,category,mechanism,image,avg_rating,category_rank,min_players,max_players,best_players,recommend_players,min_time,max_time
0,1,브라스: 버밍엄,Brass: Birmingham,14.0,3.87,Martin Wallace|Matt Tolman|Gavan Brown,"Lina Cossette, David Forest, Damien Mammoliti","브라스: 버밍엄(Brass: Birmingham)은 보드게임 종합 1위, 게임평점 ...",전략게임,산업 / 제조|나폴레옹 시대 이후|운송|이성 시대|경제|기차,핸드 관리|네트워크 및 경로 구축|수입|변화하는 게임 세팅|테크 트리 / 테크 트랙...,https://img.boardlife.co.kr/data/photo/2025/05...,8.5,"{""전략"": 1}",2.0,4.0,4.0,2.0,60.0,120.0
1,2,세티: 외계의 지성체를 찾아서,SETI: Search for Extraterrestrial Intelligence,13.0,3.79,Tomáš Holek,"Jakub Politzer, František Sedláček, Ondřej Hrd...",세티: 외계의 지성체를 찾아서(SETI: Search for Extraterrest...,전략게임,SF 공상 과학|우주 탐험,게임 종료 보너스|수입|솔로/솔로테어 게임|변화하는 게임 세팅|차례 순서: 정방향|...,https://img.boardlife.co.kr/data/photo/2026/03...,8.5,"{""전략"": 2}",1.0,4.0,3.0,2.0,40.0,160.0
2,3,글룸헤이븐,Gloomhaven,14.0,3.91,Isaac Childres,"Alexandr Elichev, Josh T. McDowell, Alvaro Nebot","글룸헤이븐(Gloomhaven)은 보드게임 종합 3위, 게임평점 8.4점, 난이도 ...","전략게임, 테마게임",모험|탐험|판타지|전투|미니어처,협력 게임|핸드 관리|조립 보드|동시 액션 선택|플레이어별 특수능력|캠페인/카드 운...,https://img.boardlife.co.kr/data/boardgame_str...,8.4,"{""전략"": 3, ""테마"": 1}",1.0,4.0,3.0,2.0,60.0,120.0


---
#### 9. 저장

In [14]:
games.to_csv('data/boardlife_games_stats_final.csv', index=False, encoding='utf-8-sig')
print(f'저장 완료: boardlife_games_stats_final.csv → {games.shape[0]:,}행 {games.shape[1]}컬럼')

저장 완료: boardlife_games_stats_final.csv → 3,322행 20컬럼
